# Scenario

Every morning a folder named DailyLogs receives server log files exported from monitoring systems.  

Before anyone analyzes the logs, my script must:  
- Create any missing folders
- Read every CSV file
- Ignore non-CSV files
- Count important statistics
- Create a backup of every CSV
- Rename the processed files
- Generate a summary report

In [7]:
# Part 1 - Create the Folder Structure

import os
import csv
import shutil
from datetime import datetime

parent_folder = "Week2 - Automating Multiple Files and Folder Management"

total_servers = 0
cpu_total = 0
down_count = 0
csv_count = 0
high_cpu_count = 0

if os.path.exists(parent_folder):
    shutil.rmtree(parent_folder)

print("Old project folder removed.")

if not os.path.exists(parent_folder):
    os.mkdir(parent_folder)


daily_logs = os.path.join(parent_folder, "DailyLogs")
reports3 = os.path.join(parent_folder, "Reports3")
backups2 = os.path.join(parent_folder, "Backups2")

for folder in [daily_logs, backups2, reports3]:
    if not os.path.exists(folder):
        os.mkdir(folder)

print("Project folders created!")

# Part 2 - Generate Sample CSV Files

sample_files = {
    "WEB01.csv": [
        {"server": "WEB01", "status": "Up", "cpu": 35},
        {"server": "WEB02", "status": "Down", "cpu": 92}
    ],
    "DB01.csv": [
        {"server": "DB01", "status": "Up", "cpu": 48},
        {"server": "DB02", "status": "Down", "cpu": 88}
    ],
    "APP01.csv": [
        {"server": "APP01", "status": "Up", "cpu": 42},
        {"server": "APP02", "status": "Up", "cpu": 55}
    ]
}

for filename, rows, in sample_files.items():
    path = os.path.join(daily_logs, filename)

    with open(path, "w", newline = "") as file:
        writer = csv.DictWriter(file, fieldnames=["server", "status", "cpu"])
        writer.writeheader()
        writer.writerows(rows)

print("Sample CSV files created.")
print()

with open(os.path.join(daily_logs, "README.txt"), "w") as file:
    file.write("Ignore this file for testing purposes.")

with open(os.path.join(daily_logs, "notes.md"), "w") as file:
    file.write("This is a markdown file.")

print("Non-CSV files created.")
print()

# Part 3 - Process Only CSV Files

for file in os.listdir(daily_logs):
    if file.endswith(".csv"):
        print("Processing", file)
print()

# Part 4 - Calcuate Statistics

for file in os.listdir(daily_logs):
    if file.endswith(".csv"):
        csv_count += 1
        path = os.path.join(daily_logs, file)

        with open(path, "r") as csv_file:
            reader = csv.DictReader(csv_file)

            for row in reader:

                total_servers += 1
                
                if row["status"] == "Down":
                    down_count += 1   

                cpu = int(row["cpu"])
                cpu_total += cpu

                if cpu > 80:
                    high_cpu_count += 1
                    print(row["server"], "has high CPU:", cpu)
                                   
print()

average_cpu = cpu_total / total_servers if total_servers > 0 else 0
print("Total servers:", total_servers)
print("Down servers:", down_count)
print("Total CPU:", cpu_total)
print("Total CPU average is:", average_cpu)
print()

# Part 5 - Backup Every CSV
source_folder = daily_logs
backup_folder = backups2

for file in os.listdir(source_folder):
    if file.endswith(".csv"):
        source = os.path.join(source_folder, file)
        destination = os.path.join(backup_folder, file)
        
        if os.path.isfile(source):
            shutil.copy(source, destination)
            print("Copied", file)

    else:
        print("Skipped directory:", file)

print()

# Part 6 - Rename Every Processed File

count = 1

files_to_rename = [
    file
    for file in os.listdir(daily_logs)
    if file.endswith(".csv") and not file.startswith("Processed_")
    
]

for file in sorted(files_to_rename):
    old_path = os.path.join(daily_logs, file)

    new_filename = f"Processed_{count:03}.csv"
    new_path = os.path.join(daily_logs, new_filename)

    os.rename(old_path, new_path)

    print(f"Renamed {file} -> {new_filename}")

    count += 1

print()

# Part 7 - Create a Summary Report

summary_path = os.path.join(reports3, "Summary_Report.txt")

with open(summary_path, "w") as report:
    report.write("========== Daily Log Summary ==========\n\n")
    report.write(f"Total CSV Files Processed: {csv_count}\n")
    report.write(f"Total Servers: {total_servers}\n")
    report.write(f"Down Servers: {down_count}\n")
    report.write(f"High CPU Servers (>80): {high_cpu_count}\n")
    report.write(f"Average CPU Usage: {average_cpu:.1f}\n")

    


print("Daily Log Summary Created at: ")
print(summary_path)



Old project folder removed.
Project folders created!
Sample CSV files created.

Non-CSV files created.

Processing APP01.csv
Processing DB01.csv
Processing WEB01.csv

DB02 has high CPU: 88
WEB02 has high CPU: 92

Total servers: 6
Down servers: 2
Total CPU: 360
Total CPU average is: 60.0

Copied APP01.csv
Copied DB01.csv
Copied WEB01.csv
Skipped directory: README.txt
Skipped directory: notes.md

Renamed APP01.csv -> Processed_001.csv
Renamed DB01.csv -> Processed_002.csv
Renamed WEB01.csv -> Processed_003.csv

Daily Log Summary Created at: 
Week2 - Automating Multiple Files and Folder Management/Reports3/Summary_Report.txt
